In [ ]:
import pandas as pd
import numpy as np

stations_df = pd.read_csv('stations.csv')
sections_df = pd.read_csv('metro_line.csv')

# 2. Create the station dictionary (Name -> ID)
# Strip whitespace to avoid issues ("  Clot " -> "Clot")
station_names = stations_df['NOM_ESTACIO'].str.strip().unique()

station_to_id = {name: i for i, name in enumerate(station_names)}
id_to_station = {i: name for name, i in station_to_id.items()}
N = len(station_names)  # Total number of unique stations (140)

print(f"{N} unique stations detected.")

# 3. Initialize the adjacency matrix with zeros
A = np.zeros((N, N), dtype=int)

# 4. Populate the matrix by iterating through the edges (sections)
for index, row in sections_df.iterrows():
    origin = str(row['NOM_ESTACIO_INI']).strip()
    destination = str(row['NOM_ESTACIO_FI']).strip()
    
    # Discard dummy nodes "Inici" (Start) and "Final" (End)
    if origin != "Inici" and destination != "Final":
        # Verify that both stations exist in our dictionary
        if origin in station_to_id and destination in station_to_id:
            i = station_to_id[origin]
            j = station_to_id[destination]
            
            # Set 1 in both directions (undirected graph)
            A[i, j] = 1
            A[j, i] = 1

print("Adjacency matrix built successfully!")
print("-" * 50)

# 5. Shortest path calculation using matrix powers
def find_shortest_path(origin_name, destination_name, matrix_A, mapping_dict):
    if origin_name not in mapping_dict or destination_name not in mapping_dict:
        return "Error: One of the stations does not exist in the database."
    
    i = mapping_dict[origin_name]
    j = mapping_dict[destination_name]
    
    # If the origin and destination are the same station
    if i == j:
        return 0, 1
        
    M = matrix_A.copy()
    k = 1  # Path length
    num_nodes = matrix_A.shape[0]
    
    # Keep multiplying until a value > 0 is found at cell (i, j)
    # Ensure we don't exceed the maximum possible number of nodes (num_nodes)
    while M[i, j] == 0 and k < num_nodes:
        k += 1
        M = M @ matrix_A  # Equivalent to computing A^k step by step
        
    if M[i, j] > 0:
        num_paths = M[i, j]
        return k, num_paths
    else:
        return -1, 0  # If we traverse the entire graph and it remains 0, they are disconnected

S'han detectat 140 estacions úniques.
Matriu d'adjacència construïda amb èxit!
--------------------------------------------------


In [ ]:
# === TESTING ZONE ===

origin_test = "Mas Blau"
destination_test = "Ciutat Meridiana"

result = find_shortest_path(origin_test, destination_test, A, station_to_id)

if isinstance(result, str):
    print(result)  # Prints the error for non-existent stations
else:
    k_min, num_routes = result
    
    print(f"Origin Station: {origin_test} (ID: {station_to_id.get(origin_test)})")
    print(f"Destination Station: {destination_test} (ID: {station_to_id.get(destination_test)})")
    
    if k_min == -1:
        print("No path found between these stations.")
    else:
        print(f"-> The SHORTEST route has a length of: {k_min} sections (stops).")
        print(f"-> There is/are {num_routes} different path(s) that allow this shortest trip.")

Estació Origen: Mas Blau (ID: 107)
Estació Destí:  Ciutat Meridiana (ID: 136)
-> El recorregut MÉS CURT té una llargada de: 28 trams (parades).
-> Hi ha 2 camí/ns diferent/s que permeten fer aquest trajecte més curt.
